# European Parliament: do MEPs vote with their country or with their party?

Main analysis notebook (report appendix). Runs top to bottom in about 5 minutes.

**Before running** (from the repo root):
```
pip install -r requirements.txt
python src/download_data.py     # HowTheyVote.eu release pinned in src/config.py
python src/cleaned_data.py      # builds data/processed/*.parquet (about 30 s)
```
All parameters are in `src/config.py`. Tables are saved to `results/tables/`, figures
to `results/figures/`.

"Party" in the title means the **European political group** (EPP, S&D, ...).
"National party" means the party an MEP was elected for (CDU, PSOE, ...).

| | Method | Code |
|---|---|---|
| Course topic 1 | Graph clustering: own Louvain + modularity, resolution sweep, NMI/AMI/ARI | `src/louvain.py`, `src/rq1.py` |
| Course topic 2 | Frequent itemsets: own A-priori and PCY | `src/frequent_items.py`, `src/rq2.py` |
| Outside technique | Ideal-point scaling with SVD of the MEP x vote matrix | `src/svd_map.py` |

In [1]:
import sys, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / "src"))
import numpy as np, pandas as pd
pd.set_option("display.precision", 3); pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

import config
from cleaned_data import load
from agreement import build_term_graph
import rq1, rq2, svd_map, frequent_items as fi, rq3_switchers as rq3, figures
from evaluation import describe_communities

OUT = Path("results"); (OUT / "tables").mkdir(parents=True, exist_ok=True)
(OUT / "figures").mkdir(parents=True, exist_ok=True)
def save(df, name, index=False):
    df.to_csv(OUT / "tables" / f"{name}.csv", index=index)
    return df

d = load()
print("HowTheyVote release:", config.RELEASE)

HowTheyVote release: 2026-11-21


## 1. Data

In [2]:
v = d["votes"]
overview = (v[v.term > 0].groupby("term")
            .agg(votes=("vote_id", "size"), contested=("contested", "sum"),
                 first=("timestamp", "min"), last=("timestamp", "max")))
print(f"{len(d['member_votes']):,} MEP-vote records, {d['member_votes'].member_id.nunique():,} MEPs, {len(v):,} votes")
print(d["member_votes"].position.value_counts().to_string())
save(overview.reset_index(), "data_overview")

18,287,530 MEP-vote records, 1,275 MEPs, 25,782 votes


position
FOR             9074187
AGAINST         6683445
DID_NOT_VOTE    1614762
ABSTENTION       915136


,term,votes,contested,first,last
0,9,18520,14128,2020-02-10 17:19:01,2024-04-25 12:58:39
1,10,6505,5872,2024-07-17 09:09:17,2026-11-12 11:57:33


## 2. Agreement graphs

For each term: contested votes only (larger side <= 90% of FOR+AGAINST), MEPs with
>= 500 FOR/AGAINST votes, agreement = share of shared votes on the same side (sparse
matrix products), then a 15-nearest-neighbour graph.

In [3]:
G = {}
for term in config.TERMS:
    t = time.perf_counter()
    G[term] = build_term_graph(d, term)
    g = G[term]
    print(f"term {term}: {g['X'].shape[0]} MEPs x {g['X'].shape[1]} votes, "
          f"{g['X'].nnz:,} records, {int((g['W'] > 0).sum() // 2):,} edges, "
          f"{int((g['W'].sum(1) == 0).sum())} isolated, {time.perf_counter() - t:.0f}s")

term 9: 780 MEPs x 14128 votes, 8,702,278 records, 8,762 edges, 0 isolated, 34s


term 10: 731 MEPs x 5872 votes, 3,469,095 records, 8,198 edges, 0 isolated, 11s


## 3. Validating our Louvain against networkx

Same graph, same 10 seeds. If our implementation is right, its partitions should agree
with networkx's about as much as networkx agrees with itself across seeds.

In [4]:
val_rows = []
for term, g in G.items():
    per_seed, summary, _ = rq1.validate_louvain(g["W"])
    save(per_seed.assign(term=term), f"louvain_validation_seeds_term{term}")
    val_rows.append(dict(term=term, **summary))
validation = save(pd.DataFrame(val_rows), "louvain_validation")
validation

,term,Q_ours,Q_ours_sd,Q_networkx,Q_networkx_sd,NMI_ours_vs_networkx,NMI_ours_vs_ours,NMI_networkx_vs_networkx,time_ours_s,time_networkx_s
0,9,0.810,6.323e-04,0.809,6.713e-04,0.972,0.982,0.967,0.138,0.067
1,10,0.845,3.555e-04,0.845,1.364e-04,0.991,0.988,0.993,0.115,0.062


## 4. RQ1: communities vs group, country and national party

Best-modularity partition over 10 seeds (gamma = 1). `NMI_chance` = NMI when the
community labels are shuffled: the baseline. AMI and ARI are about 0 by chance.
`Q_of_label_partition` = modularity if the communities were exactly the groups /
countries / parties.

In [5]:
main = []
for term, g in G.items():
    comm, runs, Q = rq1.best_of_seeds(g["W"])
    g["comm"], g["runs"] = comm, runs
    t = rq1.main_table(g, comm, runs)
    print(f"term {term}: Louvain modularity {t.attrs['Q_louvain']:.3f}, {t.attrs['n_communities']} communities")
    main.append(t.reset_index().assign(term=term, Q_louvain=t.attrs["Q_louvain"],
                                       n_communities=t.attrs["n_communities"]))
main = save(pd.concat(main), "rq1_main")
main[["term", "label", "n_labels", "NMI", "NMI_chance", "AMI", "ARI", "Q_of_label_partition",
      "AMI_min_over_seeds", "AMI_max_over_seeds"]]

term 9: Louvain modularity 0.811, 9 communities


term 10: Louvain modularity 0.845, 12 communities


,term,label,n_labels,NMI,NMI_chance,AMI,ARI,Q_of_label_partition,AMI_min_over_seeds,AMI_max_over_seeds
0,9,group,8,0.860,0.019,0.857,0.861,0.773,0.857,0.886
1,9,country,27,0.211,0.059,0.162,0.054,0.364,0.129,0.162
2,9,party,221,0.576,0.238,0.443,0.119,0.317,0.426,0.443
0,10,group,9,0.838,0.031,0.833,0.766,0.778,0.814,0.836
1,10,country,27,0.282,0.080,0.219,0.082,0.329,0.197,0.219
2,10,party,217,0.630,0.287,0.480,0.153,0.313,0.466,0.481


### Pair agreement

Mean agreement between two MEPs by whether they share a group and/or a country. The
baseline is two MEPs with neither in common.

In [6]:
pairs = pd.concat([rq1.pair_agreement(g["agree"], g["labels"]).assign(term=t) for t, g in G.items()])
save(pairs, "rq1_pair_agreement")

,pair_type,mean_agreement,sd,pairs,minus_baseline,term
0,"same group, same national party",0.975,0.044,3269,0.413,9
1,"same group, same country, different party",0.947,0.078,1180,0.384,9
2,"same group, different country",0.887,0.083,43817,0.324,9
3,"different group, same country",0.557,0.200,15868,-0.005,9
4,"different group, different country (baseline)",0.562,0.186,236208,0.000,9
0,"same group, same national party",0.984,0.016,2920,0.442,10
1,"same group, same country, different party",0.942,0.089,837,0.399,10
2,"same group, different country",0.906,0.070,37002,0.364,10
3,"different group, same country",0.530,0.251,13922,-0.013,10
4,"different group, different country (baseline)",0.543,0.244,212013,0.000,10


### What the communities contain

In [7]:
for term, g in G.items():
    desc = save(describe_communities(g["labels"], g["comm"]).assign(term=term),
                f"rq1_communities_term{term}")
    print(f"Term {term}"); display(desc)

Term 9


,community,size,main_group,main_group_share,groups,countries,parties,term
0,0,193,EPP,0.99,"EPP 192, ECR 1, ESN 0","DEU 32, POL 19, ESP 14","DEU:CDU 24, ESP:PP 14, POL:PO 13",9
1,6,140,SD,0.99,"SD 138, NI 1, RENEW 1","ESP 22, ITA 21, PRT 10","ESP:PSOE 21, ITA:PD 19, PRT:PS 10",9
2,4,113,RENEW,0.97,"RENEW 110, EPP 2, NI 1","FRA 25, ESP 10, ROU 10","FRA:LaREM/MoDEM 9, ESP:CS 7, DEU:FDP 6",9
3,7,92,GREEN_EFA,0.86,"GREEN_EFA 79, NI 13, ECR 0","DEU 28, FRA 15, ITA 12","DEU:GRÜNE. 22, FRA:Europe Écologie 13, ITA:M5S 8",9
4,3,72,ECR,0.81,"ECR 58, NI 12, EPP 2","POL 28, HUN 14, ITA 8","POL:PiS 26, HUN:Fidesz-KDNP 13, ITA:FdL 7",9
5,2,66,ID,0.76,"ID 50, ECR 9, NI 7","FRA 26, DEU 11, NLD 6","FRA:RN 26, DEU:AfD 11, SWE:SD 4",9
6,5,47,GUE_NGL,0.91,"GUE_NGL 43, NI 4, ECR 0","ESP 8, GRC 8, FRA 7","FRA:LFI 6, GRC:SYRIZA 6, DEU:DIE LINKE. 5",9
7,1,31,ID,0.94,"ID 29, ECR 1, EPP 1","ITA 31, AUT 0, BEL 0","ITA:L 29, ITA:FdL 1, ITA:SIT 1",9
8,8,26,SD,1.00,"SD 26, ECR 0, EPP 0","DEU 20, AUT 6, BEL 0","DEU:SPD 20, AUT:SPÖ 6, AUT:Die Grünen 0",9


Term 10


,community,size,main_group,main_group_share,groups,countries,parties,term
0,2,144,EPP,0.98,"EPP 141, NI 3, ECR 0","DEU 32, ROU 10, HUN 9","DEU:CDU 25, HUN:TISZA 9, ITA:FI 8",10
1,1,137,SD,0.99,"SD 136, EPP 1, ECR 0","ITA 22, ESP 20, DEU 14","ITA:PD 22, ESP:PSOE 18, DEU:SPD 14",10
2,6,78,RENEW,0.97,"RENEW 76, EPP 1, NI 1","FRA 14, DEU 10, NLD 7","FRA:Ren 6, SVK:PS 6, DEU:FDP 5",10
3,11,75,PFE,0.80,"PFE 60, ECR 11, NI 3","CZE 11, HUN 11, ITA 8","HUN:Fidesz-KDNP 10, CZE:ANO 2011 8, ITA:L 8",10
4,3,56,GREEN_EFA,0.96,"GREEN_EFA 54, GUE_NGL 1, SD 1","DEU 15, NLD 7, FRA 5","DEU:GRÜNE. 12, FRA:Les Écologistes 5, NLD:GL-P...",10
5,5,52,GUE_NGL,0.88,"GUE_NGL 46, NI 6, ECR 0","ITA 10, FRA 9, DEU 7","FRA:LFI 9, ITA:M5S 8, GRC:SYRIZA 4",10
6,10,49,ECR,0.94,"ECR 46, EPP 3, ESN 0","ITA 24, ROU 5, BEL 3","ITA:FdL 24, BEL:N-VA 3, CZE:ODS 3",10
7,4,42,ESN,0.60,"ESN 25, NI 17, ECR 0","DEU 21, SVK 8, BGR 3","DEU:AfD 15, DEU:Bündnis Sahra 5, SVK:SMER-SD 5",10
8,0,31,PFE,0.94,"PFE 29, EPP 1, ESN 1","FRA 31, AUT 0, BEL 0","FRA:RN 29, FRA:R 1, FRA:UDR 1",10
9,9,23,EPP,1.00,"EPP 23, ECR 0, ESN 0","POL 23, AUT 0, BEL 0","POL:KO 13, POL:PO 6, POL:Independent 2",10


### Where groups split, do they split along national lines?

Inside each group: how often two MEPs of the **same** national party land in the same
community, versus two MEPs of **different** national parties. If splits ignored
nationality, the two shares would be equal.

In [8]:
splits = {t: rq1.splits_follow_nationality(g["labels"], g["comm"]) for t, g in G.items()}
save(pd.concat([s.assign(term=t) for t, s in splits.items()]), "rq1_splits_nationality")
figures.splits(splits, OUT / "figures" / "splits_nationality.png")
pd.concat([s.assign(term=t) for t, s in splits.items()])

,group,size,communities_with_5plus,together_same_party,together_different_party,pairs_same_party,term
0,ECR,69,2.0,0.979,0.670,380,9
1,EPP,197,1.0,0.996,0.948,761,9
2,GREEN_EFA,79,1.0,1.000,1.000,334,9
3,GUE_NGL,43,1.0,1.000,1.000,53,9
4,ID,79,2.0,1.000,0.368,785,9
5,RENEW,111,1.0,0.947,0.983,171,9
6,SD,164,2.0,1.000,0.715,763,9
7,ALL (pooled),742,NaN,0.994,0.850,3247,9
0,ECR,79,3.0,1.000,0.321,488,10
1,EPP,192,3.0,0.993,0.544,845,10


### Resolution sweep

Higher gamma -> more, smaller communities. Which label do the communities follow at
each scale?

In [9]:
sweeps = {t: rq1.resolution_sweep(g, config.GAMMA_SWEEP) for t, g in G.items()}
save(pd.concat([s.assign(term=t) for t, s in sweeps.items()]), "rq1_resolution_sweep")
figures.resolution_sweep(sweeps, OUT / "figures" / "resolution_sweep.png")
pd.concat([s.assign(term=t) for t, s in sweeps.items()])

,gamma,n_communities,n_communities_10plus,largest_share,AMI_group,NMI_group,AMI_country,NMI_country,AMI_party,NMI_party,term
0,0.1,5,5,0.247,0.849,0.850,0.069,0.099,0.376,0.484,9
1,0.2,6,6,0.247,0.874,0.876,0.074,0.111,0.392,0.508,9
2,0.3,6,6,0.247,0.874,0.876,0.074,0.111,0.392,0.508,9
3,0.5,7,7,0.247,0.868,0.870,0.106,0.147,0.408,0.530,9
4,0.7,8,8,0.247,0.878,0.880,0.131,0.177,0.427,0.556,9
5,1.0,9,9,0.247,0.857,0.860,0.162,0.211,0.443,0.576,9
6,1.5,14,14,0.142,0.790,0.796,0.282,0.344,0.508,0.656,9
7,2.0,18,18,0.122,0.745,0.754,0.347,0.416,0.551,0.705,9
8,3.0,22,22,0.087,0.709,0.721,0.399,0.474,0.591,0.745,9
9,4.0,25,25,0.077,0.688,0.702,0.422,0.500,0.603,0.758,9


### By policy area

The graph is rebuilt from the votes of one top-level OEIL policy area at a time. Areas
with fewer than 300 contested votes in a term are skipped (too few votes per MEP).

In [10]:
policy = {t: rq1.policy_areas(d, t) for t in G}
save(pd.concat([p.assign(term=t) for t, p in policy.items()]), "rq1_policy_areas")
figures.policy_areas(policy, OUT / "figures" / "policy_areas.png")
pd.concat([p.assign(term=t) for t, p in policy.items()])[
    ["term", "label", "n_votes", "n_meps", "AMI_group", "AMI_country", "AMI_party",
     "NMI_country", "NMI_country_chance", "skipped"]]

,term,label,n_votes,n_meps,AMI_group,AMI_country,AMI_party,NMI_country,NMI_country_chance,skipped
0,9,European citizenship,907,739.0,0.881,0.114,0.417,0.157,0.049,False
1,9,"Internal market, single market",992,763.0,0.821,0.197,0.456,0.255,0.072,False
2,9,"Economic, social and territorial cohesion",3130,777.0,0.805,0.259,0.504,0.319,0.082,False
3,9,Economic and monetary system,458,712.0,0.755,0.238,0.474,0.302,0.085,False
4,9,External relations of the Union,2786,777.0,0.851,0.188,0.460,0.241,0.065,False
5,9,"Area of freedom, security and justice",786,747.0,0.858,0.183,0.444,0.233,0.063,False
6,9,State and evolution of the Union,3209,783.0,0.842,0.179,0.455,0.233,0.064,False
0,10,European citizenship,350,690.0,0.802,0.153,0.437,0.210,0.067,False
1,10,"Internal market, single market",283,NaN,NaN,NaN,NaN,NaN,NaN,True
2,10,"Economic, social and territorial cohesion",1116,728.0,0.824,0.159,0.449,0.217,0.069,False


### Robustness: number of neighbours k, and NI excluded

In [11]:
robust = save(pd.concat([rq1.robustness(g).assign(term=t) for t, g in G.items()]), "rq1_robustness")
robust

,k,subset,n_meps,n_communities,Q,AMI_group,AMI_country,AMI_party,ARI_group,ARI_country,ARI_party,term
0,10,all MEPs,780,16,0.836,0.783,0.293,0.513,0.660,0.109,0.182,9
1,10,NI excluded,742,16,0.836,0.815,0.292,0.516,0.680,0.111,0.186,9
2,15,all MEPs,780,9,0.811,0.857,0.162,0.443,0.861,0.054,0.119,9
3,15,NI excluded,742,9,0.811,0.907,0.162,0.446,0.895,0.054,0.122,9
4,25,all MEPs,780,8,0.790,0.874,0.133,0.428,0.902,0.045,0.110,9
5,25,NI excluded,742,8,0.790,0.925,0.133,0.430,0.938,0.045,0.112,9
0,10,all MEPs,731,16,0.864,0.793,0.277,0.515,0.666,0.108,0.186,10
1,10,NI excluded,701,16,0.864,0.815,0.280,0.528,0.681,0.110,0.194,10
2,15,all MEPs,731,12,0.845,0.833,0.219,0.480,0.766,0.082,0.153,10
3,15,NI excluded,701,12,0.845,0.862,0.217,0.491,0.785,0.082,0.158,10


## 5. Outside technique: SVD political map

Each MEP's position on the first two singular vectors of the centred MEP x vote matrix
(+1 FOR, -1 AGAINST, 0 missing), corrected for attendance. Silhouette measures how
well a label separates MEPs on the map; the shuffled-label baseline is the chance level.
NI is excluded from the silhouette (not a group).

In [12]:
maps, sep, centres = {}, [], []
for term, g in G.items():
    c, ex = svd_map.ideal_points(g["X"])
    c = svd_map.orient(c, g["labels"].group.to_numpy())
    g["svd"] = c
    maps[term] = (c, g["labels"], ex)
    sep.append(svd_map.separation(c, g["labels"]).assign(term=term,
               explained_dim1=ex[0], explained_dim2=ex[1]))
    centres.append(svd_map.group_centres(c, g["labels"]).reset_index().assign(term=term))
save(pd.concat(centres), "svd_group_centres")
figures.svd_maps(maps, OUT / "figures" / "svd_map.png")
save(pd.concat(sep), "svd_separation")

,label,silhouette,chance_mean,chance_p95,n_meps,n_labels,term,explained_dim1,explained_dim2
0,group,0.601,-0.083,-0.056,742,7,9,0.337,0.162
1,country,-0.303,-0.319,-0.248,742,27,9,0.337,0.162
2,party,-0.207,-0.810,-0.774,658,125,9,0.337,0.162
0,group,0.470,-0.105,-0.068,701,8,10,0.435,0.141
1,country,-0.363,-0.357,-0.278,701,27,10,0.435,0.141
2,party,-0.089,-0.831,-0.802,622,122,10,0.435,0.141


### Is the horseshoe real? Do the extremes vote together?

The map bends: the far left and far right both sit low on dimension 2. SVD/PCA produce
such an arch even when the data lie on ONE gradient (a known artefact), so we test it:
(1) how much of dimension 2 is just a parabola of dimension 1, and (2) whether The Left
and the far right oppose a united centre (EPP + S&D + Renew) together more often than
independence would predict.

In [13]:
arch = pd.DataFrame([dict(term=t, **svd_map.arch_check(g["svd"])) for t, g in G.items()])
save(arch, "svd_arch_check")
extremes = save(pd.DataFrame([svd_map.extremes_co_opposition(d, t) for t in G]), "svd_extremes_co_opposition")
display(arch); extremes

,term,R2_linear,R2_quadratic
0,9,8.486e-03,0.682
1,10,2.200e-04,0.401


,term,far_right,centre_united_share,left_opposes,far_right_opposes,both_oppose,both_if_independent,ratio_to_independent,left_far_right_majority_agreement
0,9,ID,0.617,0.376,0.729,0.181,0.274,0.659,0.264
1,10,PFE,0.765,0.329,0.763,0.134,0.251,0.532,0.179


### Did voting become more one-dimensional after the 2024 election?

Share of variance on the first SVD dimension per half-year, each window using 500
randomly drawn contested votes (5 repeats; error bars = 2 sd). Windows with fewer than
500 contested votes are skipped.

In [14]:
dims = save(svd_map.dimensionality_over_time(d), "svd_dimensionality_over_time")
figures.dimensionality(dims, OUT / "figures" / "dimensionality.png")
dims

,half,term,contested_votes,dim1_share,dim1_sd,dim2_share
0,2020-H1,9,851,0.320,0.003,0.232
1,2020-H2,9,2913,0.354,0.003,0.199
2,2021-H1,9,2686,0.384,0.009,0.163
3,2021-H2,9,1288,0.396,0.009,0.192
4,2022-H1,9,1919,0.339,0.008,0.175
5,2022-H2,9,807,0.349,0.004,0.161
6,2023-H1,9,932,0.378,0.003,0.148
7,2023-H2,9,1372,0.401,0.010,0.137
8,2024-H1,9,1360,0.332,0.004,0.163
9,2024-H2,10,450,NaN,NaN,NaN


## 6. RQ2: coalitions with our A-priori and PCY

Two kinds of baskets:
- **FOR baskets**: one per adopted contested vote; items = national parties whose
  majority voted FOR. Dense: about 55% of all parties are in every basket.
- **Rebel baskets**: one per contested vote; items = national parties whose majority voted
  against the rest of their own European group. Sparse: about 6 parties per vote.

Both algorithms are run on both and must return identical itemsets (asserted).

In [15]:
B, party_group = {}, {}
for term in config.TERMS:
    B[f"FOR, term {term}"] = fi.build_baskets(d, term, level="party", passed_only=True)[0]
    B[f"rebels, term {term}"], _, party_group[term] = rq2.rebel_baskets(d, term)
save(pd.DataFrame([dict(basket_set=k, **fi.basket_stats(v)) for k, v in B.items()]), "rq2_basket_stats")

,basket_set,baskets,distinct_items,mean_basket_size,pairs_if_counted_naively,triples_if_counted_naively
0,"FOR, term 9",7602,143,80.389,25348259,719256605
1,"rebels, term 9",14128,131,6.397,450365,2129223
2,"FOR, term 10",2284,139,76.485,6823391,180522980
3,"rebels, term 10",5872,127,5.114,122544,479822


### A-priori vs PCY

`pcy_candidate_share` = PCY's candidate pairs as a share of A-priori's. Pass-2 memory:
A-priori keeps one 4-byte counter per pair of frequent items (triangular matrix); PCY
keeps a bitmap plus 12 bytes per candidate pair, so PCY only saves memory when it
removes more than about two thirds of the pairs.

In [16]:
cmp = pd.concat([
    rq2.compare_algorithms({k: v for k, v in B.items() if k.startswith("FOR")}, [0.5, 0.7]),
    rq2.compare_algorithms({k: v for k, v in B.items() if k.startswith("rebels")}, [0.01, 0.02])])
save(cmp, "rq2_apriori_vs_pcy")

,baskets,support,n_baskets,mean_basket_size,frequent_items,frequent_pairs,frequent_triples,apriori_pair_candidates,pcy_pair_candidates,pcy_candidate_share,frequent_bucket_share,apriori_pass2_bytes,pcy_pass2_bytes,apriori_s,pcy_s
0,"FOR, term 9",0.50,7602,80.389,95,3111,40915,4465,3624,0.812,0.340,17860,44739,1.476,1.730
1,"FOR, term 9",0.70,7602,80.389,57,515,2386,1596,784,0.491,0.120,6384,10659,0.232,0.792
2,"FOR, term 10",0.50,2284,76.485,85,2419,35641,3570,2838,0.795,0.270,14280,35307,0.394,0.652
3,"FOR, term 10",0.70,2284,76.485,57,544,1820,1596,808,0.506,0.103,6384,10947,0.076,0.217
0,"rebels, term 9",0.01,14128,6.397,110,760,429,5995,1015,0.169,0.083,23980,13431,0.300,0.513
1,"rebels, term 9",0.02,14128,6.397,90,185,74,4005,259,0.065,0.021,16020,4359,0.154,0.321
2,"rebels, term 10",0.01,5872,5.114,101,378,198,5050,512,0.101,0.041,20200,7395,0.090,0.135
3,"rebels, term 10",0.02,5872,5.114,82,111,17,3321,139,0.042,0.012,13284,2919,0.060,0.121


In [17]:
# Number of PCY buckets: too few -> every bucket is frequent and PCY prunes nothing
sweep_pcy = pd.concat([rq2.bucket_sweep(B["rebels, term 9"], 0.01).assign(baskets="rebels, term 9, s=0.01"),
                       rq2.bucket_sweep(B["FOR, term 9"], 0.5).assign(baskets="FOR, term 9, s=0.5")])
save(sweep_pcy, "rq2_pcy_bucket_sweep")

,n_buckets,frequent_bucket_share,pcy_pair_candidates,pairs_of_frequent_items,bitmap_bytes,pass2_bytes,apriori_pass2_bytes,baskets
0,101,1.000,5995,5995,13,71953,23980,"rebels, term 9, s=0.01"
1,1009,0.961,5844,5995,127,70255,23980,"rebels, term 9, s=0.01"
2,10007,0.083,1015,5995,1251,13431,23980,"rebels, term 9, s=0.01"
3,100003,0.008,1118,5995,12501,25917,23980,"rebels, term 9, s=0.01"
0,101,1.000,4465,4465,13,53593,17860,"FOR, term 9, s=0.5"
1,1009,1.000,4465,4465,127,53707,17860,"FOR, term 9, s=0.5"
2,10007,0.340,3624,4465,1251,44739,17860,"FOR, term 9, s=0.5"
3,100003,0.032,3475,4465,12501,54201,17860,"FOR, term 9, s=0.5"


### Who rebels together?

All pairs of parties that rebelled together at least 30 times. Lift = how many times
more often they rebel together than if they rebelled independently (baseline 1).

In [18]:
rebel_summary, rebel_pairs = [], {}
for term in config.TERMS:
    summ, pairs = rq2.rebel_pair_summary(B[f"rebels, term {term}"], party_group[term])
    rebel_summary.append(summ.assign(term=term)); rebel_pairs[term] = pairs
    save(pairs, f"rq2_rebel_pairs_term{term}")
save(pd.concat(rebel_summary), "rq2_rebel_pair_summary")

,pair_type,pairs,co_rebellions,median_lift,share_lift_above_2,share_of_co_rebellions,term
0,"different group, different country",2592,241209,1.347,0.261,0.593,9
1,"different group, same country",147,16074,1.640,0.388,0.039,9
2,same group,1058,149787,3.552,0.804,0.368,9
0,"different group, different country",679,35717,1.601,0.330,0.487,10
1,"different group, same country",48,2445,1.593,0.375,0.033,10
2,same group,463,35144,4.186,0.892,0.479,10


In [19]:
for term, pairs in rebel_pairs.items():
    print(f"Term {term}: most frequent rebel pairs")
    display(pairs.sort_values("count", ascending=False).head(10)[["itemset", "groups", "count", "support", "lift"]])

Term 9: most frequent rebel pairs


,itemset,groups,count,support,lift
584,FRA:RN + ITA:L,ID+ID,1671,0.118,4.592
230,FRA:PS + FRA:Place publique,SD+SD,1090,0.077,8.018
611,DNK:V + NLD:VVD,RENEW+RENEW,1025,0.073,4.442
887,CZE:ANO 2011 + DEU:FDP,RENEW+RENEW,1022,0.072,3.442
724,AUT:FPÖ + BEL:Vlaams Belang,ID+ID,1012,0.072,3.987
1131,BEL:N-VA + CZE:ODS,ECR+ECR,1007,0.071,2.787
1056,AUT:FPÖ + DEU:AfD,ID+ID,992,0.070,2.990
322,DEU:FDP + DEU:Freie Wähler,RENEW+RENEW,907,0.064,6.710
1112,CZE:ANO 2011 + NLD:VVD,RENEW+RENEW,889,0.063,2.848
660,DEU:FDP + NLD:VVD,RENEW+RENEW,852,0.060,4.247


Term 10: most frequent rebel pairs


,itemset,groups,count,support,lift
504,FRA:Les Républicains + SVN:SDS,EPP+EPP,486,0.083,2.652
519,FRA:Les Républicains + NLD:BBB,EPP+EPP,434,0.074,2.574
497,NLD:BBB + SVN:SDS,EPP+EPP,327,0.056,2.700
358,FRA:IDL + SWE:SD,ECR+ECR,327,0.056,3.361
801,DEU:Freie Wähler + FRA:Les Républicains,RENEW+EPP,322,0.055,1.586
17,FRA:PS + FRA:Place publique,SD+SD,298,0.051,16.404
266,DEU:FDP + DEU:Freie Wähler,RENEW+RENEW,270,0.046,4.272
445,BEL:N-VA + SWE:SD,ECR+ECR,269,0.046,2.916
92,FIN:VAS + SWE:V,GUE_NGL+GUE_NGL,264,0.045,7.834
339,DEU:Freie Wähler + NLD:VVD,RENEW+RENEW,258,0.044,3.527


### Which national-party coalitions win votes?

Our A-priori on the FOR baskets (one per adopted contested vote; items = national parties
whose majority voted FOR), support >= 0.5, up to 3 parties. A frequent itemset is a coalition
of parties that was on the winning side together on at least half of the adopted votes.
*Cross-border* = parties from 2+ countries; *cross-group* = parties from 2+ European groups.
Cross-border is almost automatic (two parties are rarely from the same country), so
cross-group is the share that matters. The group mix shows which groups the 3-party
coalitions come from in each term (ID and PfE both written FAR).

In [ ]:
win_items, win_summary, win_mix = rq2.winning_party_coalitions(d, support=0.5, max_size=3)
top_border, top_group = rq2.top_coalitions(win_items, size=2, top=10)
save(win_summary, "rq2_winning_coalitions_summary")
save(win_mix, "rq2_winning_coalitions_group_mix")
save(pd.concat([top_border.assign(kind="cross-border"), top_group.assign(kind="cross-group")]),
     "rq2_winning_coalitions_top_pairs")
figures.winning_coalitions(win_mix, path=OUT / "figures" / "winning_coalitions.png")
display(win_summary, win_mix.head(10), top_group)

,term,size,coalitions,cross_border,cross_group
0,9,2,3111,0.961,0.721
1,9,3,40915,0.998,0.882
2,10,2,2419,0.959,0.709
3,10,3,35641,0.998,0.888


term,groups,share_term9,share_term10,change
0,EPP+SD,0.111,0.203,0.093
1,EPP+RENEW,0.104,0.142,0.038
2,EPP+RENEW+SD,0.059,0.117,0.059
3,RENEW+SD,0.166,0.112,-0.054
4,EPP,0.055,0.069,0.014
5,EPP+GREEN_EFA+SD,0.016,0.062,0.046
6,GREEN_EFA+SD,0.081,0.049,-0.033
7,EPP+GREEN_EFA,0.010,0.048,0.038
8,GREEN_EFA+RENEW+SD,0.081,0.047,-0.033
9,EPP+GREEN_EFA+RENEW,0.006,0.034,0.028


,term,support,lift,coalition
33,9,0.830,1.040,ESP:PSOE (SD) + FRA:LaREM/MoDEM (RENEW)
34,9,0.829,1.030,FRA:LaREM/MoDEM (RENEW) + ITA:PD (SD)
37,9,0.827,1.041,ESP:PSOE (SD) + FRA:MoDem (RENEW)
38,9,0.826,1.031,FRA:MoDem (RENEW) + ITA:PD (SD)
41,9,0.818,1.036,FRA:LaREM/MoDEM (RENEW) + HRV:SDP (SD)
43,9,0.817,1.037,FRA:LaREM/MoDEM (RENEW) + PRT:PS (SD)
46,9,0.815,1.038,FRA:MoDem (RENEW) + HRV:SDP (SD)
47,9,0.815,1.039,FRA:MoDem (RENEW) + PRT:PS (SD)
48,9,0.813,1.038,FRA:LaREM/MoDEM (RENEW) + NLD:PvdA (SD)
50,9,0.810,1.039,FRA:MoDem (RENEW) + NLD:PvdA (SD)


### Group coalitions on adopted votes, term 9 vs term 10

Frequent group itemsets (A-priori, support >= 0.3) on adopted contested votes; support of
each coalition in both terms. ID (term 9) and PfE (term 10) are both written FAR.

In [20]:
coalitions = save(rq2.group_coalitions(d), "rq2_group_coalitions")
coalitions

,itemset,size,support_term9,lift_term9,support_term10,lift_term10,change
0,GUE_NGL + SD,2,0.695,1.078,0.608,1.131,-0.087
1,GREEN_EFA + GUE_NGL,2,0.682,1.181,0.606,1.219,-0.076
2,GUE_NGL + RENEW,2,0.655,1.019,0.582,1.075,-0.073
3,GREEN_EFA + GUE_NGL + SD,3,0.667,1.286,0.595,1.400,-0.072
4,GUE_NGL + RENEW + SD,3,0.639,1.108,0.573,1.238,-0.066
5,GREEN_EFA + GUE_NGL + RENEW,3,0.624,1.207,0.572,1.336,-0.052
6,GREEN_EFA + GUE_NGL + RENEW + SD,4,0.614,1.324,0.565,1.545,-0.049
7,RENEW + SD,2,0.828,1.033,0.811,1.102,-0.017
8,GREEN_EFA + SD,2,0.780,1.082,0.768,1.138,-0.012
9,GREEN_EFA + RENEW,2,0.738,1.026,0.751,1.105,0.014


In [21]:
# Answer key (term 10): are HowTheyVote's required group combinations inside our baskets?
check, required = rq2.validate_against_required(d)
print(check)
save(required, "rq2_required_coalitions_term10").head(12)

{'votes': 1921, 'required_rows': 14802, 'share_required_combos_inside_basket': 0.7786109985137144}


,required_groups,votes,share_of_votes,support_in_our_baskets
0,EPP,994,0.517,0.866
1,EPP+SD,745,0.388,0.724
2,EPP+RENEW,691,0.360,0.757
3,RENEW+SD,691,0.360,0.802
4,SD,515,0.268,0.846
5,EPP+GREEN_EFA,509,0.265,0.655
6,GREEN_EFA+SD,486,0.253,0.760
7,ECR+EPP,428,0.223,0.457
8,ECR+RENEW+SD,351,0.183,0.326
9,GUE_NGL+SD,330,0.172,0.590


### The EPP as pivot

On contested votes where the left bloc (S&D, Greens, Left) and the right bloc (ECR,
ID/PfE, ESN) majorities disagree: which side the EPP majority takes, and how often the
right side wins with and without the EPP. Then: the share of votes each group could
flip on its own (simple majority of FOR vs AGAINST; abstentions ignored).

In [22]:
pivot_terms, pivot_half = rq2.epp_pivot(d)
power = rq2.voting_power(d)
save(pivot_terms, "rq2_epp_pivot"); save(pivot_half, "rq2_epp_pivot_by_half")
save(power, "rq2_voting_power")
figures.epp_pivot(pivot_half, power, OUT / "figures" / "epp_pivot.png")
display(pivot_terms); power

,subset,term,votes,epp_with_right,right_wins_when_epp_joins,right_wins_when_epp_against
0,all contested votes,9,10654.0,0.447,0.518,0.002
1,all contested votes,10,4608.0,0.288,0.810,0.003
2,final votes (is_main),9,629.0,0.203,0.516,0.000
3,final votes (is_main),10,325.0,0.203,0.879,0.000
4,legislation (COD),9,2270.0,0.513,0.624,0.004
5,legislation (COD),10,669.0,0.329,0.891,0.002


,group,pivotal_share_term9,pivotal_share_term10,mean_voting_members_term9,mean_voting_members_term10
0,ECR,0.078,0.092,52.234,63.845
1,EPP,0.245,0.549,161.294,157.315
2,ESN,NaN,0.028,NaN,21.587
3,FAR,0.063,0.085,54.383,65.588
4,GREEN_EFA,0.150,0.049,65.391,45.461
5,GUE_NGL,0.076,0.036,33.313,36.650
6,RENEW,0.221,0.071,91.946,66.086
7,SD,0.243,0.255,129.528,114.350


## 7. RQ3: did group switchers vote differently before they switched?

Only votes before the switch. Switchers are compared with colleagues in the same origin
group over the same window who never left it. AUC = P(switcher scores higher than a
stayer); 0.5 = no difference. 95% CI by bootstrapping over clusters of MEPs who switched
together. Same-group "switches" (split membership records) are dropped.

In [23]:
events, per_event, rq3_summary = rq3.run(d)
print(f"{len(events)} switch events; {(per_event.status == 'ok').sum()} with >= "
      f"{config.MIN_PRE_SWITCH_VOTES} pre-switch votes")
save(per_event, "rq3_events")
save(rq3_summary, "rq3_summary")

140 switch events; 118 with >= 200 pre-switch votes


,test,events,clusters,AUC,CI_low,CI_high,share_events_above_half,by_term
0,leave,93,68,0.642,0.521,0.737,0.624,"{9: 0.637, 10: 0.671}"
1,join,53,46,0.742,0.659,0.815,0.792,"{9: 0.709, 10: 0.825}"


In [24]:
(per_event[per_event.status == "ok"]
 .groupby(["term", "origin", "destination"])
 .agg(events=("member_id", "size"), leave_auc=("leave_auc", "mean"), join_auc=("join_auc", "mean"))
 .sort_values("events", ascending=False).head(10))

events  leave_auc  join_auc
term origin    destination                             
9    ID        NI               28      0.578       NaN
     EPP       NI               13      0.920       NaN
     SD        NI                9      0.697       NaN
10   NI        ECR               5        NaN     0.807
9    NI        GREEN_EFA         4        NaN     0.902
               EPP               4        NaN     0.870
               ECR               4        NaN     0.568
     GREEN_EFA NI                3      0.490       NaN
     EPP       RENEW             3      0.794     0.901
     SD        RENEW             3      0.509     0.528

## 8. RQ1 before and after 2019: 2014-2019 vs 2019-2024 and 2024-2026

The same RQ1 analysis and SVD map on the 2014-2019 term, from the VoteWatch Europe data
(`src/load_votewatch.py`), next to the two HowTheyVote terms. Each vote uses the European
group the MEP was in on that day, from the EP Open Data API (`src/fetch_ep_groups.py`).
National-party results are shown for 2019-2026 only: VoteWatch gives one end-of-term party
label per MEP. Run the two scripts once before this section; terms 9 and 10 are read from the
tables saved above.

In [ ]:
import ep8_analysis
ep8 = ep8_analysis.run()
ami, pairs_3, sep_3, splits_3, summary_3 = ep8_analysis.compare_terms(ep8)
ep8_analysis.three_term_figure(OUT / "figures" / "three_terms_rq1.png")
display(summary_3, ami, pairs_3, sep_3, splits_3)